<a href="https://colab.research.google.com/github/charlesfsouza/r_sepse_dbt_pipeline/blob/main/SimulandoDadosSepse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Simulação dados de sepse


In [ ]:
# Pacotes
library("dplyr")

In [ ]:
# Parametros constantes

## Tamanho amostra
c_tamanho_amostra = 2000

## Numero de medicos que atua no CTI

c_numero_medicos <- trunc(c_tamanho_amostra ^ (1/4),0)

## Numero de medicacoes disponiveis

c_numero_medicacoes <- 30

## Numero de medicacoes indicadas para sepse

c_numero_medicacoes_sepse <- 5


## Prevalencia de Sepse
#Prevalência de sepse grave e choque séptico: 29,6%
#Local: 229 UTIs em todas as regiões do Brasil
#Detalhes: Estudo multicêntrico nacional com corte transversal.
#Fonte: ILAS – SPREAD Study [ilas.org.br]

c_prev_sepse = .296

## Data hora inicial
c_dat_hora_inicial = strptime("2024-01-01 00:00:00","%Y-%m-%d %H:%M:%S")

## Data hora inicial
c_dat_hora_final = strptime("2024-12-31 23:59:59","%Y-%m-%d %H:%M:%S")



## CQH, 2011b - 5,3 dias, com uma variação de 2,4 a 15,1 dias
c_min_tempo_permanencia = 2.4
c_max_tempo_permanencia = 15.1


## Prescricao de um item de medicacao

c_prob_medicacao_sepse_nao = .50
c_prob_medicacao_sepse_sim = .90

c_media_itens_medicacao_prescritos = 5

## Prescricao de um item de exame de laboratorio

c_prob_exame_laboratorio_sepse_nao = .25
c_prob_exame_laboratorio_sepse_sim = .90

c_media_itens_exame_laboratorio_prescritos = 3


In [ ]:
# Parametros vetores

## Perfil etario dos pacientes
### Perfil de pacientes de uma unidade de terapia intensiva de adultos de um município paraibano1
v_dsc_faixa_etaria <- c("18 a 28 anos", "29 a 39 anos", "40 a 50 anos", "51 a 60 anos",
                  "61 a 70 anos", "71 a 80 anos", "81 a 90 anos", "91 a 100 anos")

v_id_faixa_etaria <- seq(1:length(v_dsc_faixa_etaria))

v_proporcao_faixa_etaria <- c(.038, .02, .103, .106, .212, .243, .209, .066)


v_dsc_sexo <- c("Feminino","Masculino")
v_id_sexo <- seq(1,length(v_dsc_sexo))
v_proporcao_sexo <- c(.508, .492)



In [ ]:
# Variaveis de atendimento

set.seed(123)


## Atendimento

# id_atendimento <- seq(1:c_tamanho_amostra)
dat_atendimento <- seq(from = c_dat_hora_inicial, to = c_dat_hora_final, by = "10 min") %>%
  sample(c_tamanho_amostra)


tempo_permanencia_em_dias <- seq(c_min_tempo_permanencia,c_max_tempo_permanencia,by = .1) %>%
  sample(c_tamanho_amostra,replace = T)

dat_alta <- dat_atendimento + tempo_permanencia_em_dias * 24 * 60 * 60


## Medicos

id_medico_atendimento <- sample(
  seq(1:c_numero_medicos)
  ,c_tamanho_amostra
  , replace = TRUE)






In [ ]:
# Variaveis de paciente
id_paciente <- seq(1:c_tamanho_amostra)

id_faixa_etaria_paciente <- sample(
  v_id_faixa_etaria
  ,c_tamanho_amostra
  ,replace = TRUE
  , prob = v_proporcao_faixa_etaria
  )

id_sexo_paciente <- sample(
  v_id_sexo
  ,c_tamanho_amostra
  ,replace = TRUE
  , prob = v_proporcao_sexo
)



In [ ]:
# Variaveis de avaliacao

ind_sepse <- sample(
  c(0,1)
  ,c_tamanho_amostra
  , prob = c(1-c_prev_sepse,c_prev_sepse)
  ,replace = TRUE
)

## Protocolo Sepse
qtd_inclusoes_protocolo_sepse <- 0

for (i in 1:length(tempo_permanencia_em_dias)) {

# Parâmetros da distribuição Gamma
shape <- 1
scale <- tempo_permanencia_em_dias[i]

tmp <- rgamma(1000, shape = shape, scale = scale)

qtd_inclusoes_protocolo_sepse[i] <- tmp[tmp >= 1 & tmp <= scale] %>%
  sample(1) %>%
  trunc(0)

}

qtd_inclusoes_protocolo_sepse <- qtd_inclusoes_protocolo_sepse * ind_sepse


In [ ]:
# data frame com dados do atendimento
#df <- data.frame(id_atendimento, dat_atendimento, dat_alta, tempo_permanencia_em_dias, ind_sepse,qtd_inclusoes_protocolo_sepse, id_paciente,id_faixa_etaria_paciente,id_sexo_paciente)
df_atendimento <- data.frame( dat_atendimento, dat_alta, id_medico = id_medico_atendimento,id_paciente) %>%
  arrange(dat_atendimento) %>%
  mutate(
    id_atendimento = row_number()
    ) %>%
  relocate(id_atendimento,1)

df_atendimento %>% head()


,id_atendimento,dat_atendimento,dat_alta,id_medico,id_paciente
,<int>,<dttm>,<dttm>,<int>,<int>
1,1,2024-01-01 00:40:00,2024-01-10 22:16:00,3,1571
2,2,2024-01-01 05:00:00,2024-01-05 12:12:00,4,192
3,3,2024-01-01 06:40:00,2024-01-04 06:40:00,3,29
4,4,2024-01-01 09:00:00,2024-01-04 23:24:00,2,1645
5,5,2024-01-01 12:30:00,2024-01-04 10:06:00,2,509
6,6,2024-01-01 13:00:00,2024-01-13 15:24:00,4,293


In [ ]:
# data frame com dados do paciente
df_paciente <- data.frame(id_paciente, id_faixa_etaria_paciente, id_sexo_paciente)


In [ ]:
## Função para contar avaliações da equipe medica que ocorrem diariamente às 7h
#contar_avaliacoes <- function(inicio, fim) {
#  # Gera sequência de dias às 7h
#  avaliacoes <- seq(
#    from = lubridate::floor_date(inicio, unit = "day") + lubridate::hours(7),
#    to = lubridate::floor_date(fim, unit = "day") + lubridate::hours(7),
#    by = "1 day"
#  )
#  # Filtra avaliações dentro do intervalo real
#  sum(avaliacoes >= inicio & avaliacoes <= fim)
#}
#
#

In [ ]:
## Aplica a função linha a linha
#df <- df %>%
#  rowwise() %>%
#  mutate(qtd_avaliacoes = contar_avaliacoes(dat_atendimento, dat_alta)) %>%
#  relocate(qtd_avaliacoes, .before = qtd_inclusoes_protocolo_sepse) %>%
#  ungroup()

In [ ]:
# Gera datas/hora de avaliacao
 # partindo da definicao que cada paciente
 # é avaliado pela equipe medica diariamente às 7h


df_avaliacoes <- data.frame()

for (i in 1:c_tamanho_amostra) {

  tmp_id_atendimento <- df_atendimento$id_atendimento[i]
  tmp_dat_atendimento <- df_atendimento$dat_atendimento[i]
  tmp_dat_alta <- df_atendimento$dat_alta[i]
  tmp_ind_sepse <- ind_sepse[i]

  tmp_datas_avaliacao <- seq(
      from = lubridate::floor_date(tmp_dat_atendimento, unit = "day") + lubridate::hours(7)
      ,to = lubridate::floor_date(tmp_dat_alta, unit = "day") + lubridate::hours(7)
      ,by = "day"
  )

  tmp_datas_avaliacao <- tmp_datas_avaliacao[tmp_datas_avaliacao >= tmp_dat_atendimento & tmp_datas_avaliacao <= tmp_dat_alta]

  step1_tmp_amostra_marcacao_sepse <- c(1,sample(c(0,1),max(0,length(tmp_datas_avaliacao)-1),replace = TRUE))

  step2_tmp_amostra_marcacao_sepse <- tmp_ind_sepse * sample(step1_tmp_amostra_marcacao_sepse,length(step1_tmp_amostra_marcacao_sepse))

  tmp_df <- data.frame(
    id_atendimento = rep(tmp_id_atendimento,length(tmp_datas_avaliacao))
    ,dat_avaliacao = tmp_datas_avaliacao
    ,ind_marcacao_sepse = step2_tmp_amostra_marcacao_sepse
  )

df_avaliacoes <- tmp_df %>%
  bind_rows(df_avaliacoes)

}


id_medico_avaliacao <- sample(seq(1:c_numero_medicos),nrow(df_avaliacoes), replace = TRUE)

df_avaliacoes <- df_avaliacoes %>%
  arrange(
    dat_avaliacao,id_atendimento
  ) %>%
  mutate(
    id_avaliacao = row_number()
    ,id_medico = id_medico_avaliacao
  ) %>%
relocate(id_avaliacao,1) %>%
relocate(id_medico,.before = dat_avaliacao)

head(df_avaliacoes)







,id_avaliacao,id_atendimento,id_medico,dat_avaliacao,ind_marcacao_sepse
,<int>,<int>,<int>,<dttm>,<dbl>
1,1,1,2,2024-01-01 07:00:00,1
2,2,2,2,2024-01-01 07:00:00,1
3,3,3,4,2024-01-01 07:00:00,0
4,4,1,2,2024-01-02 07:00:00,1
5,5,2,2,2024-01-02 07:00:00,1
6,6,3,1,2024-01-02 07:00:00,0


In [ ]:
 # data frame de medicacoes

 # Cria vetor com nomes das medicações
v_medicacoes <- c(
  "Midazolam", "Fentanila", "Tramadol", "Diazepam", "Haloperidol",
  "Amiodarona", "Dobutamina", "Dopamina", "Adrenalina", "Atropina",
  "Bicarbonato de sódio", "Glicose 50%", "Gluconato de cálcio",
  "Hidrocortisona", "Manitol", "Etomidato",
  "Ceftriaxona", "Meropeném", "Vancomicina", "Piperacilina-Tazobactam"
)

# Define quais são antibióticos
v_antibioticos <- c("Ceftriaxona", "Meropeném", "Vancomicina", "Piperacilina-Tazobactam")

# Cria data frame
df_medicacao <- data.frame(
  nome_medicacao = v_medicacoes
  ,ind_antibiotico = ifelse(v_medicacoes %in% v_antibioticos,1,0)
) %>%
  arrange(nome_medicacao) %>%
  mutate(
    id_medicacao = row_number()
  ) %>%
  relocate(id_medicacao,1)

head(df_medicacao)

,id_medicacao,nome_medicacao,ind_antibiotico
,<int>,<chr>,<dbl>
1,1,Adrenalina,0
2,2,Amiodarona,0
3,3,Atropina,0
4,4,Bicarbonato de sódio,0
5,5,Ceftriaxona,1
6,6,Diazepam,0


In [ ]:
# Data frame de exames laboratoriais


# Vetor com nomes dos exames
v_exame_laboratorio <- c(
  "Hemograma completo", "Gasometria arterial", "Lactato", "PCR (Proteína C reativa)",
  "Procalcitonina", "Função renal (ureia e creatinina)", "Função hepática (TGO, TGP, bilirrubinas)",
  "Eletrólitos (Na, K, Cl)", "Glicemia", "Coagulograma", "Cultura de sangue",
  "Cultura de urina", "Cultura de secreção traqueal", "Cultura de ferida", "Sumário de urina",
  "Albumina", "D-dímero", "Troponina", "CK-MB", "Velocidade de hemossedimentação (VHS)"
)

# Define quais são recomendados em sepse
v_exames_laboratorio_sepse <- c(
  "Hemograma completo", "Gasometria arterial", "Lactato", "PCR (Proteína C reativa)",
  "Procalcitonina", "Função renal (ureia e creatinina)", "Função hepática (TGO, TGP, bilirrubinas)",
  "Eletrólitos (Na, K, Cl)", "Coagulograma", "Cultura de sangue", "Cultura de urina",
  "Cultura de secreção traqueal", "Albumina"
)

# Cria data frame
df_exame_laboratorio <- data.frame(
  nome_exame = v_exame_laboratorio
) %>%
  arrange(nome_exame) %>%
  mutate(
    id_exame_laboratorio = row_number()
  ) %>%
  relocate(id_exame_laboratorio,1)

head(df_exame_laboratorio)


,id_exame_laboratorio,nome_exame
,<int>,<chr>
1,1,Albumina
2,2,CK-MB
3,3,Coagulograma
4,4,Cultura de ferida
5,5,Cultura de sangue
6,6,Cultura de secreção traqueal


In [ ]:
# data frame de itens de prescricao

df_itens_prescricao <- df_medicacao %>%
  select(
    nome_medicacao
    ,ind_antibiotico
  ) %>%
  rename(
    nome_item = nome_medicacao
  ) %>%
  mutate(
    tipo_item = ifelse(ind_antibiotico == 1,"Antibiotico","Medicacao")
  ) %>%
  select(-ind_antibiotico) %>%
  bind_rows(
    df_exame_laboratorio %>%
      select(nome_exame) %>%
      rename(nome_item = nome_exame) %>%
      mutate(
        tipo_item = "Exame Laboratorio"
      )
  ) %>%
  arrange(nome_item) %>%
  mutate(id_item_prescricao = row_number()) %>%
  relocate(id_item_prescricao,1)


head(df_itens_prescricao)



,id_item_prescricao,nome_item,tipo_item
,<int>,<chr>,<chr>
1,1,Adrenalina,Medicacao
2,2,Albumina,Exame Laboratorio
3,3,Amiodarona,Medicacao
4,4,Atropina,Medicacao
5,5,Bicarbonato de sódio,Medicacao
6,6,CK-MB,Exame Laboratorio


In [ ]:
# Criando prescricoes

ind_prescricao_medicacao <- 0
ind_prescricao_exame_laboratorio <- 0
ind_prescricao <- 0

for (i in 1:nrow(df_avaliacoes)) {


 ind_prescricao_medicacao[i] <- case_when(
  df_avaliacoes$ind_marcacao_sepse[i] == 0 ~
    sample(
      c(0,1)
      ,1
      ,prob = c(1-c_prob_medicacao_sepse_nao,c_prob_medicacao_sepse_nao)
      ,replace = TRUE
    )
  ,df_avaliacoes$ind_marcacao_sepse[i] == 1 ~
    sample(
        c(0,1)
        ,1
        ,prob = c(1-c_prob_medicacao_sepse_sim,c_prob_medicacao_sepse_sim)
        ,replace = TRUE
      )
  ,TRUE ~ 0
 )

 ind_prescricao_exame_laboratorio[i] <- case_when(
  df_avaliacoes$ind_marcacao_sepse[i] == 0 ~
    sample(
      c(0,1)
      ,1
      ,prob = c(1-c_prob_exame_laboratorio_sepse_nao,c_prob_exame_laboratorio_sepse_nao)
      ,replace = TRUE
    )
  ,df_avaliacoes$ind_marcacao_sepse[i] == 1 ~
    sample(
        c(0,1)
        ,1
        ,prob = c(1-c_prob_exame_laboratorio_sepse_sim,c_prob_exame_laboratorio_sepse_sim)
        ,replace = TRUE
      )
  ,TRUE ~ 0
 )

ind_prescricao[i] <- ifelse(
  ind_prescricao_medicacao[i] ==1 |
  ind_prescricao_exame_laboratorio[i] == 1
  ,1,0
    )

}





In [ ]:
# Criando quantidade de itens prescritos

# Quantidade de itens de medicacao prescritos

dist_gamma_medicacao <- rgamma(1000, shape = .2, scale = 2)
dist_gamma_medicacao <- dist_gamma_medicacao[dist_gamma_medicacao >=1 & dist_gamma_medicacao <= nrow(df_medicacao)]

qtd_itens_medicacao_prescritos <- 0

for (i in 1:nrow(df_avaliacoes)) {

  qtd_itens_medicacao_prescritos[i] <- dist_gamma_medicacao %>%
  sample(1) %>%
  trunc(0)

}

qtd_itens_medicacao_prescritos <- qtd_itens_medicacao_prescritos * ind_prescricao_medicacao

# Quantidade de itens de exame de laboratorio prescritos

dist_gamma_exame_laboratorio <- rgamma(1000, shape = .4, scale = 4)
dist_gamma_exame_laboratorio <- dist_gamma_exame_laboratorio[dist_gamma_exame_laboratorio >=1 & dist_gamma_exame_laboratorio <= nrow(df_exame_laboratorio)]

qtd_itens_exame_laboratorio_prescritos <- 0

for (i in 1:nrow(df_avaliacoes)) {

  qtd_itens_exame_laboratorio_prescritos[i] <- dist_gamma_exame_laboratorio %>%
  sample(1) %>%
  trunc(0)

}

qtd_itens_exame_laboratorio_prescritos <- qtd_itens_exame_laboratorio_prescritos * ind_prescricao_exame_laboratorio



In [ ]:
df_avaliacoes %>% head()

,id_avaliacao,id_atendimento,id_medico,dat_avaliacao,ind_marcacao_sepse
,<int>,<int>,<int>,<dttm>,<dbl>
1,1,1,2,2024-01-01 07:00:00,1
2,2,2,2,2024-01-01 07:00:00,1
3,3,3,4,2024-01-01 07:00:00,0
4,4,1,2,2024-01-02 07:00:00,1
5,5,2,2,2024-01-02 07:00:00,1
6,6,3,1,2024-01-02 07:00:00,0


In [ ]:
# Criando data frame de prescricoes


raw_prescricoes <- df_avaliacoes %>%
  select(
    id_avaliacao
    ,id_atendimento
    ,id_medico
    ,dat_avaliacao
  ) %>%
  mutate(
    ind_prescricao = ind_prescricao
    ,qtd_itens_medicacao_prescritos = qtd_itens_medicacao_prescritos
    ,qtd_itens_exame_laboratorio_prescritos = qtd_itens_exame_laboratorio_prescritos
  )


 int_prescricoes <- raw_prescricoes %>%
  filter(
    ind_prescricao == 1
  ) %>%
  rename(dat_prescricao = dat_avaliacao) %>%
  mutate(
    id_prescricao = row_number()
  ) %>%
  relocate(id_prescricao,1)

df_prescricoes <- int_prescricoes %>%
  select(
    -ind_prescricao
    ,-qtd_itens_medicacao_prescritos
    ,-qtd_itens_exame_laboratorio_prescritos
    )

head(int_prescricoes);head(df_prescricoes)



,id_prescricao,id_avaliacao,id_atendimento,id_medico,dat_prescricao,ind_prescricao,qtd_itens_medicacao_prescritos,qtd_itens_exame_laboratorio_prescritos
,<int>,<int>,<int>,<int>,<dttm>,<dbl>,<dbl>,<dbl>
1,1,1,1,2,2024-01-01 07:00:00,1,9,7
2,2,2,2,2,2024-01-01 07:00:00,1,4,4
3,3,3,3,4,2024-01-01 07:00:00,1,1,2
4,4,4,1,2,2024-01-02 07:00:00,1,1,1
5,5,5,2,2,2024-01-02 07:00:00,1,2,1
6,6,6,3,1,2024-01-02 07:00:00,1,0,2


,id_prescricao,id_avaliacao,id_atendimento,id_medico,dat_prescricao
,<int>,<int>,<int>,<int>,<dttm>
1,1,1,1,2,2024-01-01 07:00:00
2,2,2,2,2,2024-01-01 07:00:00
3,3,3,3,4,2024-01-01 07:00:00
4,4,4,1,2,2024-01-02 07:00:00
5,5,5,2,2,2024-01-02 07:00:00
6,6,6,3,1,2024-01-02 07:00:00


In [ ]:
#data frame de itens prescritos

df_itens_prescritos <- data.frame()

list_antibiotico_disp <-   df_itens_prescricao %>%
    filter(tipo_item %in% c("Antibiotico")) %>%
    pull(id_item_prescricao)

for (i in 1:nrow(int_prescricoes)) {

id = int_prescricoes$id_prescricao[i]
qtd_medicacao = int_prescricoes$qtd_itens_medicacao_prescritos[i]
qtd_exame_laboratorio = int_prescricoes$qtd_itens_exame_laboratorio_prescritos[i]

tmp_tipo_medicacao <- c('Antibiotico','Antibiotico','Antibiotico','Antibiotico','Antibiotico')

tmp_tipo_medicacao <- sample(
    c("Medicacao","Antibiotico")
    ,qtd_medicacao
    ,prob = c(.25,1-.25)
    ,replace = TRUE
)


tpm_antibioticos <- sample(
  list_antibiotico_disp
  ,min(length(list_antibiotico_disp),length(tmp_tipo_medicacao[tmp_tipo_medicacao=="Antibiotico"]))
  ,replace = FALSE
)

tmp_medicamentos <- sample(
  df_itens_prescricao %>%
    filter(tipo_item %in% c("Medicacao")) %>%
    pull(id_item_prescricao)
  ,length(tmp_tipo_medicacao[tmp_tipo_medicacao=="Medicacao"]) + length(tmp_tipo_medicacao[tmp_tipo_medicacao=="Antibiotico"]) -length(tpm_antibioticos)
  ,replace = FALSE
)


tmp_exames_laboratorio <- sample(
  df_itens_prescricao %>%
    filter(tipo_item %in% c("Exame Laboratorio")) %>%
    pull(id_item_prescricao)
  ,qtd_exame_laboratorio
  ,replace = FALSE
)

tmp_df_itens_prescritos <- data.frame(
  id_prescricao = id
  ,id_item = c(tmp_medicamentos,tpm_antibioticos,tmp_exames_laboratorio)
)

df_itens_prescritos <- df_itens_prescritos %>%
  bind_rows(tmp_df_itens_prescritos)

}

df_itens_prescritos <- df_itens_prescritos %>%
  arrange(id_prescricao,id_item) %>%
  mutate(
    id_item_prescricao = row_number()
  ) %>%
  relocate(
    id_item_prescricao,1
  )



head(df_itens_prescritos); tail(df_itens_prescritos)



,id_item_prescricao,id_prescricao,id_item
,<int>,<int>,<int>
1,1,1,7
2,2,1,8
3,3,1,10
4,4,1,15
5,5,1,18
6,6,1,20


,id_item_prescricao,id_prescricao,id_item
,<int>,<int>,<int>
39325,39325,11969,35
39326,39326,11969,38
39327,39327,11969,39
39328,39328,11970,34
39329,39329,11970,39
39330,39330,11971,31


In [ ]:
  df_itens_prescricao %>%
    filter(tipo_item %in% c("Antibiotico"))

id_item_prescricao,nome_item,tipo_item
<int>,<chr>,<chr>
7,Ceftriaxona,Antibiotico
31,Meropeném,Antibiotico
34,Piperacilina-Tazobactam,Antibiotico
39,Vancomicina,Antibiotico


In [ ]:
int_prescricoes %>% filter(id_prescricao == 530)
df_itens_prescritos %>% filter(id_prescricao == 530)

df_itens_prescricao %>% filter(id_item_prescricao %in% df_itens_prescritos$id_item[df_itens_prescritos$id_prescricao == 530])

id_prescricao,id_avaliacao,id_atendimento,id_medico,dat_prescricao,ind_prescricao,qtd_itens_medicacao_prescritos,qtd_itens_exame_laboratorio_prescritos
<int>,<int>,<int>,<int>,<dttm>,<dbl>,<dbl>,<dbl>
530,753,102,3,2024-01-20 07:00:00,1,1,1


id_item_prescricao,id_prescricao,id_item
<int>,<int>,<int>
1798,530,11
1799,530,31


id_item_prescricao,nome_item,tipo_item
<int>,<chr>,<chr>
11,Cultura de secreção traqueal,Exame Laboratorio
31,Meropeném,Antibiotico


In [ ]:
int_prescricoes %>% arrange(desc(qtd_itens_medicacao_prescritos))




id_prescricao,id_avaliacao,id_atendimento,id_medico,dat_prescricao,ind_prescricao,qtd_itens_medicacao_prescritos,qtd_itens_exame_laboratorio_prescritos
<int>,<int>,<int>,<int>,<dttm>,<dbl>,<dbl>,<dbl>
1,1,1,2,2024-01-01 07:00:00,1,9,7
20,30,2,6,2024-01-04 07:00:00,1,9,1
225,321,29,6,2024-01-12 07:00:00,1,9,0
279,401,68,3,2024-01-13 07:00:00,1,9,0
332,482,59,6,2024-01-15 07:00:00,1,9,0
352,507,85,4,2024-01-15 07:00:00,1,9,7
357,512,90,4,2024-01-15 07:00:00,1,9,1
436,626,66,5,2024-01-18 07:00:00,1,9,0
608,871,127,3,2024-01-22 07:00:00,1,9,1


In [ ]:
ch <- ls()

In [ ]:
data.frame( variavel = ch) %>% filter(stringr::str_detect(variavel,'df'))
head(df_paciente)


variavel
<chr>
df_atendimento
df_avaliacoes
df_exame_laboratorio
df_itens_prescricao
df_itens_prescritos
df_medicacao
df_paciente
df_prescricoes
tmp_df


,id_paciente,id_faixa_etaria_paciente,id_sexo_paciente
,<int>,<int>,<int>
1,1,6,2
2,2,8,1
3,3,6,2
4,4,6,1
5,5,8,2
6,6,7,2


In [ ]:
#Vetores com nomes masculinos e femininos e sobrenomes
# Nomes Femininos (Top 50 aproximado)
nomes_femininos <- c(
# Top 20
  "Helena", "Cecília", "Maite", "Alice", "Laura", "Maria Cecília",
  "Maria Alice", "Aurora", "Antonella", "Isis", "Sophia", "Isabella",
  "Manuela", "Luísa", "Valentina", "Eloá", "Lívia", "Júlia",
  "Maria Luiza", "Heloísa",
  # Nomes Adicionais (21 ao 100)
  "Ayla", "Esther", "Melissa", "Leticia", "Maria Clara", "Sarah",
  "Rebeca", "Ana Liz", "Lorena", "Maria Júlia", "Catarina", "Lavínia",
  "Giovanna", "Beatriz", "Mariana", "Lara", "Jade", "Yasmin",
  "Agatha", "Alícia", "Alana", "Maria Eduarda", "Marina", "Aylla",
  "Clara", "Maria Liz", "Bella", "Hadassa", "Vitória", "Ana Clara",
  "Liz", "Ana Luiza", "Clarice", "Milena", "Gabriela", "Rafaela",
  "Mayla", "Mariah", "Maria Isis", "Serena", "Zoe", "Maria Flor",
  "Maria Vitória", "Isabelly", "Ana Cecília", "Melina", "Diana",
  "Luara", "Mirella", "Melinda", "Emilly", "Ana Beatriz", "Ana",
  "Maria Laura", "Bianca", "Laís", "Louise", "Chloe", "Pérola",
  "Maria Elisa", "Isabel", "Maria Heloísa", "Betina", "Eva",
  "Malu", "Isadora", "Olívia", "Elisa"
)


# Nomes Masculinos (Top 50 aproximado)
nomes_masculinos <- c(
  # Top 20
  "Miguel", "Gael", "Ravi", "Theo", "Heitor", "Arthur",
  "Noah", "Davi", "Bernardo", "Samuel", "Anthony", "Benício",
  "Gabriel", "Isaac", "Joaquim", "Matheus", "Benjamin", "Levi",
  "Pedro", "Matteo",
  # Nomes Adicionais (21 ao 100)
  "Lucas", "Nicolas", "Rafael", "Lucca", "Henrique", "Murilo",
  "Caleb", "João Miguel", "Bryan", "Henry", "Daniel", "Bento",
  "Lorenzo", "Vicente", "João Pedro", "Emanuel", "Guilherme",
  "Leonardo", "Felipe", "Anthony Gabriel", "Mathias", "Davi Lucca",
  "Pietro", "Pedro Henrique", "Gustavo", "Augusto", "João Lucas",
  "Oliver", "Ravi Lucca", "Thomas", "João", "Eduardo", "Antônio",
  "Liam", "Apollo", "Asafe", "Nathan", "Caio", "José Rael",
  "Davi Lucas", "Yuri", "Francisco", "Otávio", "Kauan", "Enzo Gabriel",
  "Vitor", "Theodoro", "Valentim", "Yan", "João Guilherme", "Vinícius",
  "Ryan", "Otto", "Dante", "José Miguel", "Enrico", "Luan", "Dominic",
  "Abner", "Enzo", "Josué", "Dom", "Endrick", "Thiago", "Léo",
  "Kalel", "Arthur Miguel", "Erick", "Davi Miguel", "Gael Henrique",
  "Raul", "Israel", "Anthony Noah", "Anthony Miguel"
)

sobrenomes_populares_100 <- c(
    # Top 50 (Mais Frequentes)
    "Silva", "Santos", "Oliveira", "Souza", "Pereira", "Costa",
    "Rodrigues", "Almeida", "Nascimento", "Lima", "Gomes", "Ferreira",
    "Carvalho", "Ribeiro", "Martins", "Araújo", "Melo", "Barbosa",
    "Pinto", "Vieira", "Freitas", "Fernandes", "Dias", "Campos",
    "Andrade", "Moraes", "Reis", "Nunes", "Machado", "Azevedo",
    "Cavalcanti", "Rocha", "Pires", "Aires", "Pacheco", "Gonçalves",
    "Lopes", "Castro", "Afonso", "Teixeira", "Fonseca", "Duarte",
    "Neves", "Brito", "Borges", "Correia", "Vasconcelos", "Leal",
    "Miranda", "Nogueira",

    # Nomes Adicionais (51 a 100)
    "Moreira", "Godoy", "Siqueira", "Monteiro", "Moura", "Cardoso",
    "Viana", "Cunha", "Tavares", "Mendes", "Pinheiro", "Ramos",
    "Coelho", "Zanetti", "Cordeiro", "Freire", "Xavier", "Antunes",
    "Farias", "Teles", "Pires", "Garcia", "Bastos", "Sanches",
    "Peixoto", "Soares", "Marques", "Porto", "Pires", "Sampaio",
    "Vaz", "Rosa", "Matos", "Lemos", "Guerra", "Bueno",
    "Passos", "Magalhães", "Menezes", "França", "Dantas", "Sá",
    "Pinto", "Nobrega", "Vargas", "Vidal", "Pessoa", "Rangel",
    "Franco", "Couto"
)

In [ ]:
# FUNCAO PARA GERAR NOMES ALEATOREAMENTE A PARTIR DAS LISTAS ANTERIORES

gerar_nome <- function(id_sexo,qtd_maxima_sobrenome){

nome <- ""

if (id_sexo == 2){

nome <- paste(
  c(
    sample(nomes_masculinos,1) # seleciona de forma aleatoria um primeiro nome
    ,sample(
      sobrenomes_populares_100 # seleciona de forma aleatoria 1,2 ou 3 sobrenomes sem reposicao
      ,sample(
        seq(1,qtd_maxima_sobrenome)
        ,1
      )
      ,replace = FALSE
    )
  ),collapse = " ")

}
else {

nome <- paste(
  c(
    sample(nomes_femininos,1) # seleciona de forma aleatoria um primeiro nome
    ,sample(
      sobrenomes_populares_100 # seleciona de forma aleatoria 1,2 ou 3 sobrenomes sem reposicao
      ,sample(
        seq(1,qtd_maxima_sobrenome)
        ,1
      )
      ,replace = FALSE
    )
  ),collapse = " ")

}

nome <- iconv(
  nome,
  from = "UTF-8",
  to = "ASCII//TRANSLIT" # O segredo está aqui!
) %>%
  toupper()

return(nome)

}




In [ ]:
ls() %>%
  data.frame() %>%
  rename(variavel = 1) %>%
  filter(stringr::str_detect(variavel,"df"))

variavel
<chr>
df_atendimento
df_avaliacoes
df_exame_laboratorio
df_itens_prescricao
df_itens_prescritos
df_medicacao
df_paciente
df_prescricoes
tmp_df


In [ ]:
# gerando tabelas

# paciente
tab_paciente <- df_paciente %>%
  rowwise() %>%
  mutate(
    nome_paciente = gerar_nome(id_sexo_paciente,3)
  ) %>%
  ungroup()

# sexo
tab_sexo <- data.frame(v_id_sexo,v_dsc_sexo) %>%
  rename(
    id_sexo = 1
    ,dsc_sexo = 2
  )

# faixa etaria
tab_faixa_etaria <- data.frame(v_id_faixa_etaria,v_dsc_faixa_etaria) %>%
  rename(
    id_faixa_etaria = 1
    ,dsc_faixa_etaria = 2
  )

# medico
tab_medico <- data.frame(id_medico = df_atendimento$id_medico) %>%
  unique() %>%
  rowwise() %>%
  mutate(
    nome_medico = gerar_nome(sample(c(1,2),1),3)
  )

# atendimento
tab_atendimento <- df_atendimento


# avaliacao
tab_avaliacao <- df_avaliacoes

# prescricao
tab_prescricao <- df_prescricoes

# itens
tab_item <- df_itens_prescricao %>%
  rename(
    id_item = id_item_prescricao
  )

# item prescricao
tab_item_prescricao <- df_itens_prescritos


In [ ]:
ls() %>%
  data.frame() %>%
  rename(variavel = 1) %>%
  filter(stringr::str_detect(variavel,"tab"))

variavel
<chr>
tab_atendimento
tab_avaliacao
tab_faixa_etaria
tab_item
tab_item_prescricao
tab_medico
tab_paciente
tab_prescricao
tab_sexo


In [ ]:
# exportando arquivos para posteriormente carregar no bigquery

write.csv(tab_atendimento, "tab_atendimento.csv", row.names = FALSE)
write.csv(tab_avaliacao, "tab_avaliacao.csv", row.names = FALSE)
write.csv(tab_faixa_etaria, "tab_faixa_etaria.csv", row.names = FALSE)
write.csv(tab_item, "tab_item.csv", row.names = FALSE)
write.csv(tab_item_prescricao, "tab_item_prescricao.csv", row.names = FALSE)
write.csv(tab_paciente, "tab_paciente.csv", row.names = FALSE)
write.csv(tab_prescricao, "tab_prescricao.csv", row.names = FALSE)
write.csv(tab_sexo, "tab_sexo.csv", row.names = FALSE)
write.csv(tab_medico, "tab_medico.csv", row.names = FALSE)

In [ ]:
head(tab_paciente)

id_paciente,id_faixa_etaria_paciente,id_sexo_paciente,nome_paciente
<int>,<int>,<int>,<chr>
1,6,2,KAUAN MELO VIEIRA RANGEL
2,8,1,CECILIA ANDRADE FARIAS RIBEIRO
3,6,2,LEVI PEIXOTO
4,6,1,MARIA LIZ FRANCA
5,8,2,ANTHONY XAVIER
6,7,2,ANTONIO ARAUJO ZANETTI
